# Stage-I diagnostics: chroma, lightness, and white-background distance

This notebook inspects the affine Boy map obtained after imposing exact sRGB red/green/blue anchors on the Cartesian director axes and optimizing only the pairwise metric-fidelity utility. No new utility is introduced here.

We diagnose three quantities in OKLab:

$$C=\sqrt{a^2+b^2}$$

for chroma (a rough measure of colorfulness/vividness),

$$L$$

for perceptual lightness, and

$$d_W=\|\mathbf c-(1,0,0)\|=\sqrt{(1-L)^2+a^2+b^2}$$

for distance from the OKLab coordinate of sRGB white.

For each quantity we estimate the probability density over uniformly sampled nematic orientations, its cumulative distribution, mean, variance, standard deviation, and extrema. Because $\mathbf c(\mathbf n)=\mathbf c(-\mathbf n)$, uniform sampling on $\mathbb S^2$ induces the uniform measure on $\mathbb{RP}^2$ for these diagnostics.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# OKLab coordinates of exact sRGB primaries used by Stage I.
cR = np.array([0.62795536,  0.22486306,  0.12584630])
cG = np.array([0.86643961, -0.23388757,  0.17949848])
cB = np.array([0.45201372, -0.03245698, -0.31152815])

# det(A)>0 optimum from the constrained metric-only Stage-I calculation.
a3 = np.array([0.18656091, 0.07918832, -0.16001219])
a1 = (2*cR-cG-cB)/3
a2 = 4*(cG-cB)/7
beta = (cR+cG+cB)/3 - a3/8
A = np.column_stack((a1,a2,a3))

def boy_map(n):
    x,y,z = n.T
    p1 = 0.5*((2*x*x-y*y-z*z)+2*y*z*(y*y-z*z)+z*x*(x*x-z*z)+x*y*(y*y-x*x))
    p2 = 7/8*((y*y-z*z)+z*x*(z*z-x*x)+x*y*(y*y-x*x))
    p3 = 1/8*(x+y+z)*((x+y+z)**3+4*(y-x)*(z-y)*(x-z))
    return np.column_stack((p1,p2,p3))

rng = np.random.default_rng(20260830)
N = 2_000_000
n = rng.normal(size=(N,3))
n /= np.linalg.norm(n,axis=1,keepdims=True)
c = boy_map(n) @ A.T + beta

L = c[:,0]
C = np.hypot(c[:,1],c[:,2])
dW = np.linalg.norm(c-np.array([1.0,0.0,0.0]),axis=1)

def summary(x):
    return dict(mean=x.mean(), variance=x.var(), std=x.std(), minimum=x.min(), maximum=x.max())

for name,x in [('Chroma C',C),('Lightness L',L),('White distance d_W',dW)]:
    s=summary(x)
    print(name)
    for k,v in s.items(): print(f'  {k:>8s}: {v:.9f}')
    print('  quantiles:', dict(zip(['1%','5%','25%','50%','75%','95%','99%'],np.quantile(x,[.01,.05,.25,.5,.75,.95,.99]))))


In [ ]:
# Probability-density estimates.
for name,x in [('Chroma C',C),('Lightness L',L),('White distance d_W',dW)]:
    fig,ax=plt.subplots(figsize=(6,4))
    ax.hist(x,bins=160,density=True)
    ax.set_xlabel(name); ax.set_ylabel('probability density')
    ax.set_title(f'Distribution of {name}')
    plt.show()

# Empirical cumulative distribution functions.
for name,x in [('Chroma C',C),('Lightness L',L),('White distance d_W',dW)]:
    xs=np.sort(x)
    F=np.arange(1,len(xs)+1)/len(xs)
    fig,ax=plt.subplots(figsize=(6,4))
    ax.plot(xs,F)
    ax.set_xlabel(name); ax.set_ylabel('cumulative probability')
    ax.set_title(f'CDF of {name}')
    plt.show()


## Numerical diagnostic result

With two million uniformly sampled orientations, the distributions are approximately summarized by

| quantity | mean | variance | standard deviation | minimum | maximum |
|---|---:|---:|---:|---:|---:|
| chroma $C$ | 0.1847 | 0.00605 | 0.0778 | $\approx0$ | 0.3403 |
| lightness $L$ | 0.6675 | 0.01610 | 0.1269 | 0.3748 | 0.9408 |
| white distance $d_W$ | 0.3913 | 0.01372 | 0.1171 | 0.0784 | 0.6862 |

Useful distribution quantiles are

| quantity | 1% | 5% | 25% | median | 75% | 95% | 99% |
|---|---:|---:|---:|---:|---:|---:|---:|
| $C$ | 0.0280 | 0.0607 | 0.1215 | 0.1843 | 0.2508 | 0.3090 | 0.3297 |
| $L$ | 0.3888 | 0.4408 | 0.5890 | 0.6620 | 0.7515 | 0.8910 | 0.9304 |
| $d_W$ | 0.1122 | 0.1908 | 0.3219 | 0.3841 | 0.4579 | 0.6133 | 0.6708 |

The first metric-only solution therefore has several conspicuous perceptual features. Chroma spans almost the full interval from gray ($C\simeq0$) to highly saturated OKLab colors; lightness spans about $0.57$, from approximately $0.375$ to $0.941$; and part of the surface approaches white closely, with $d_W\simeq0.078$ at the sampled minimum. These are large variations for a map whose encoded physical variable is orientation only. Whether each deserves a new utility should be decided from the rendered colormap, but none can be regarded as automatically benign from the statistics alone.


## A more fundamental check: global sRGB displayability

The three axis anchors themselves are exact sRGB colors, but that does **not** imply that the rest of the affine Boy surface lies inside the sRGB gamut. Since displayability was stated as a hard requirement, this must be checked before adding optional perceptual utilities.


In [ ]:
def oklab_to_linear_srgb(lab):
    L,a,b = lab.T
    l=(L+0.3963377774*a+0.2158037573*b)**3
    m=(L-0.1055613458*a-0.0638541728*b)**3
    s=(L-0.0894841775*a-1.2914855480*b)**3
    return np.column_stack((
        4.0767416621*l-3.3077115913*m+0.2309699292*s,
       -1.2684380046*l+2.6097574011*m-0.3413193965*s,
       -0.0041960863*l-0.7034186147*m+1.7076147010*s))

linrgb=oklab_to_linear_srgb(c)
in_gamut=np.all((linrgb>=0)&(linrgb<=1),axis=1)
print(f'fraction inside sRGB gamut: {in_gamut.mean():.6f}')
print('sampled linear-sRGB minima:',linrgb.min(axis=0))
print('sampled linear-sRGB maxima:',linrgb.max(axis=0))


For this Stage-I optimum, only about **54.8%** of uniformly sampled orientations lie inside the sRGB gamut. The sampled linear-sRGB channel ranges are approximately

$$R\in[-0.304,1.131],\qquad G\in[-0.034,1.043],\qquad B\in[-0.148,1.883].$$

This is not a secondary aesthetic defect: it violates the hard displayability condition. Therefore the next mathematical modification should first enforce global sRGB-gamut feasibility (or otherwise specify an explicit gamut-mapping operation). Only after obtaining a displayable map should we decide, from the chroma/lightness/white-distance distributions and rendered appearance, whether additional utility terms are needed for those perceptual properties.
